In [25]:
# pip install langchain_huggingface
# !pip install langchain_community

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import faiss
import spacy
import re
import contractions
from textblob import TextBlob
from langchain_google_genai import ChatGoogleGenerativeAI
import os

### 1 LOAD THE DOCUMENT

In [27]:
data = open('Data1.txt').read()

### 2 TEXT NORMALIZATION

### CONVERTING ALL THE CHAR INTO LOWERCASE

In [28]:
data = data.lower()

### REMOVING THE EXTRA SPACE

In [29]:
data = re.sub(r'\s{2,}','',data)

### REMOVE NUMBERS LIKE 1.,2.

In [ ]:
data = re.sub(r'\d+\.','',data)

### CONTRACTION

In [31]:
data= contractions.fix(data)
# print(type(v))

### REMOVING SPECIAL CHARACTER AND PUNCTUATION

In [32]:
data = re.sub(r'[^0-9a-zA-Z\s]','',data)
data

'the quick brown fox jumps over the lazy dognatural language processing nlp is a fascinating field of artificial intelligence that focuses on enabling computersto understand interpret and generate human languagein a meaningful and useful way nlp combines computational linguisticsrulebased modeling of human languagewithstatistical machine learning and deep learning models to processand analyze large amounts of natural language dataone of the most important tasks in nlp is text preprocessing which involves several steps tokenizationbreaking down text into individual words phrases or sentences for example do not worry becomes do nt worrylowercasingconverting all uppercase letters to lowercase for consistency hello world  hello worldremoving special characterspunctuationeliminating unnecessary symbols hello world  hello worldstopword removalfiltering out common words like the a is and or but in on at etcstemmingreducing words to their root form running runs ran  runlemmatizationconverting 

#### TEXTBLOB

In [33]:
values = TextBlob(data).correct()  # is for correction of words


### SPACY LEMMATIZATION -> CONVERTS THE WORDS INTO ROOT FORM

In [34]:
nlp = spacy.load("en_core_web_sm")
tokens = nlp(data)
updated_tokens = [token.lemma_ for token in tokens if not token.is_stop]
data = ' '.join(updated_tokens).strip()   # strip remove extra space and char also
data

'quick brown fox jump lazy dognatural language processing nlp fascinating field artificial intelligence focus enable computersto understand interpret generate human languagein meaningful useful way nlp combine computational linguisticsrulebase modeling human languagewithstatistical machine learning deep learning model processand analyze large amount natural language dataone important task nlp text preprocessing involve step tokenizationbreake text individual word phrase sentence example worry not worrylowercasingconverte uppercase letter lowercase consistency hello world   hello worldremove special characterspunctuationeliminating unnecessary symbol hello world   hello worldstopword removalfiltering common word like etcstemmingreduce word root form run run run   runlemmatizationconverte word dictionarycanonical form well   good run   runmodern nlp application include machine translation google translate sentiment analysis analyze emotion review namedentity recognition neridentifye peop

In [35]:
tokens.ents

(google, google, gpt4, typosextra, 12345, 20240115)

### CHUNKING(CONVERTING DOC - > CHUNKS)

In [36]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 40
)
chunks =   splitter.create_documents([data])
print(type(chunks))


<class 'list'>


In [37]:
# print(chunks[0].page_content)
# print(chunks)
# (chunks[0]).metadata ={'file_name':'data.txt'}
# chunks

## embeddingss   (convert chunks to vectors)

In [38]:
chunks

[Document(metadata={}, page_content='quick brown fox jump lazy dognatural language processing nlp fascinating field artificial intelligence focus enable computersto understand interpret generate human languagein meaningful useful way nlp'),
 Document(metadata={}, page_content='languagein meaningful useful way nlp combine computational linguisticsrulebase modeling human languagewithstatistical machine learning deep learning model processand analyze large amount natural'),
 Document(metadata={}, page_content='processand analyze large amount natural language dataone important task nlp text preprocessing involve step tokenizationbreake text individual word phrase sentence example worry not'),
 Document(metadata={}, page_content='word phrase sentence example worry not worrylowercasingconverte uppercase letter lowercase consistency hello world   hello worldremove special characterspunctuationeliminating unnecessary symbol'),
 Document(metadata={}, page_content='unnecessary symbol hello world

In [40]:
embedding_model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-mpnet-base-v2'
)


'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.

In [ ]:
vectordb = FAISS.from_documents(
    documents= chunks,
    embedding=embedding_model
)
vectordb

In [ ]:
user_query = 'What is nlp?'
r_chunks = vectordb.similarity_search(user_query)

In [ ]:
r_chunks = [chunk.page_content for chunk in r_chunks]



In [ ]:
updated_r_chunks= set()

for chunk in r_chunks:
    updated_r_chunks.add(chunk.page_content)
updated_r_chunks  
R_text ='\n'.join(updated_r_chunks)
R_text  

'namedentity recognition neridentifye people place organization question answer system text summarization machine readingcomprehension chatbotsvirtual assistant like alexa siri google assistant large\nlanguagein meaningful useful way nlp combine computational linguisticsrulebase modeling human languagewithstatistical machine learning deep learning model processand analyze large amount natural\nquick brown fox jump lazy dognatural language processing nlp fascinating field artificial intelligence focus enable computersto understand interpret generate human languagein meaningful useful way nlp\ncomprehensive approach ensure clean standardized datum machine learning model \n printfgenerate uncleaned datum lenuncleaneddatasplit word'

In [ ]:
def rag_query(query,k=3):
    R_chunks = vectordb.similarity_search(query,k=k)
    # print(R_chunks)
    R_chunks = {doc.page_content for doc in R_chunks}
    R_text = '\n'.join(R_chunks)
    return R_text


def g_text(rag_query,query):
    import os
    prompt = f''' 
                 You're an helpful assistant
                 Assigned Task for you : Structure my output =>{rag_query}
                 for this input => {query}
                 Note: 
                 Don't add extra contents just structure mentioned output,
                 If there is mistake in output correct or else keep the original output
                 with strctured result.
                 output structure:
                 input:{query}
                 output: Structured output
        
        '''
        
        
    llm_model  =ChatGoogleGenerativeAI(
        model = "gemini-3.5-flash",  # gemini-3.6-flash,gemini-2.5-pro,gemini-2.5-flash
        api_key = os.environ['GeminiAPI'],
        temperature = 0.0,
        
    )   
    response = llm_model.invoke(user_prompt).content[0]['text']
    return response   
    

# print(response)
user_prompt = 'Explain nlp ?'
user_prompt = re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)
r_response =rag_query(user_prompt)
g_response = g_text(r_response,user_prompt)
print(g_response)

# response = rag_query(user_prompt)
# print(response)

NameError: name 're' is not defined